# Eval + Generate — fable200m (Colab)

Loads the trained checkpoint, generates fables from `(character, moral)` seeds,
computes reference-free metrics (Distinct-1/2, Self-BLEU, Flesch) and a 4-axis
LLM-as-judge, then writes `eval_summary.json`.

```bash
colab new -s eval --gpu T4
colab upload -s eval scripts/metrics.py /content/scripts/metrics.py
colab exec -s eval -f notebooks/eval_gen_fable200m_colab.ipynb
colab download -s eval /content/drive/MyDrive/fable200m/eval_summary.json ./results/
colab stop -s eval
```


In [ ]:
!uv pip install transformers datasets tokenizers accelerate
import sys, json, re, os
sys.path.insert(0, '/content')
from scripts.metrics import aggregate_metrics
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
MODEL_DIR = '/content/drive/MyDrive/fable200m'
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.float16).to('cuda')
gen = pipeline('text-generation', model=model, tokenizer=tok, device=0)

In [ ]:
PREFIX = '<char> {character} </char>\n<moral> {moral} </moral>\n<story>\n'
def generate_fable(character, moral, max_new=300, temp=0.9):
    prompt = PREFIX.format(character=character, moral=moral)
    out = gen(prompt, max_new_tokens=max_new, do_sample=True,
              temperature=temp, top_p=0.9, repetition_penalty=1.3)
    text = out[0]['generated_text']
    story = text.split('<story>', 1)[-1].split('</story>')[0].strip()
    return story

SEEDS = [
  {'character': 'a clever fox',      'moral': 'cleverness beats brute force'},
  {'character': 'a brave little mouse','moral': 'kindness returns to those who give it'},
  {'character': 'a proud lion',       'moral': 'pride comes before a fall'},
  {'character': 'an honest ant',     'moral': 'hard work pays off'},
]
gen_stories = [generate_fable(s['character'], s['moral']) for s in SEEDS]
for s, st in zip(SEEDS, gen_stories):
    print('###', s); print(st); print()

In [ ]:
m = aggregate_metrics(gen_stories)
print('metrics:', m)

In [ ]:
JUDGE = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
jtok = AutoTokenizer.from_pretrained(JUDGE)
jmodel = AutoModelForCausalLM.from_pretrained(JUDGE, torch_dtype=torch.float16).to('cuda')
jpipe = pipeline('text-generation', model=jmodel, tokenizer=jtok, device=0)

AXES = ['grammar', 'creativity', 'moral_clarity', 'prompt_adherence']
def judge_story(story, prompt):
    sys_p = ('You are a strict judge of children\'s fables. Rate the STORY 0-10 on four axes: '
             'grammar, creativity, moral_clarity, prompt_adherence. Respond ONLY with JSON '
             '{"grammar":int,"creativity":int,"moral_clarity":int,"prompt_adherence":int}.')
    messages = [{'role': 'system', 'content': sys_p},
                {'role': 'user', 'content': f'REQUEST:\n{prompt}\n\nSTORY:\n{story}'}]
    out = jpipe(messages, max_new_tokens=120, do_sample=False)[0]['generated_text'][-1]['content']
    mm = re.search(r'\{[^{}]*\}', out)
    if mm:
        try:
            return json.loads(mm.group(0))
        except Exception:
            pass
    return {a: 0 for a in AXES}

results = []
for s, st in zip(SEEDS, gen_stories):
    prompt = PREFIX.format(**s)
    sc = judge_story(st, prompt)
    sc['overall'] = round(sum(sc.get(a, 0) for a in AXES) / len(AXES), 2)
    results.append({'seed': s, 'story': st, 'scores': sc})
print(json.dumps(results, indent=2))

In [ ]:
summary = {'model': 'fable200m-200M', 'metrics': m, 'results': results}
with open('/content/drive/MyDrive/fable200m/eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('wrote eval_summary.json')